In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

In [0]:
control_table = "formula1_inc.control.batch_control"

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
if v_batch_id:
    delta_table = DeltaTable.forName(spark,control_table)
    source_df = (
            spark.createDataFrame([Row(batch_id=v_batch_id,status = "completed")])
                 .withColumn("updated_timestamp",current_timestamp())
            )
    (
        delta_table.alias("t")
               .merge(source_df.alias("s"),"t.batch_id = s.batch_id and t.status = 'in_progress'")
               .whenMatchedUpdate(
                   set= {"t.status": "s.status",
                         "t.updated_timestamp" : "s.updated_timestamp"}
               )
               .execute()
               )
    print(f"Batch ID {v_batch_id} is completed")
else:
    raise Exception("Batch id is missing")